## Model Comparision

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# 1. Create synthetic binary classification data
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    n_redundant=2, random_state=42
)

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF Kernel)": SVC(probability=True, kernel='rbf', random_state=42)
}

# 4. Prepare plots
plt.figure(figsize=(12, 5))

# --- ROC Curve ---
plt.subplot(1, 2, 1)
for name, model in models.items():
    model.fit(X_train, y_train)
    y_scores = model.predict_proba(X_test)[:, 1]  # Probabilities for positive class
    
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')

# --- Precision-Recall Curve ---
plt.subplot(1, 2, 2)
for name, model in models.items():
    y_scores = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    pr_auc = average_precision_score(y_test, y_scores)
    plt.plot(recall, precision, lw=2, label=f"{name} (AUC = {pr_auc:.3f})")

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')

plt.tight_layout()
plt.show()


## Manual computation

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Example: true labels and predicted probabilities
y_true = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
y_scores = np.array([0.05, 0.1, 0.4, 0.35, 0.8, 0.7, 0.2, 0.9, 0.6, 0.3])

# 1. Sort by predicted score (descending)
desc_score_indices = np.argsort(-y_scores)
y_true_sorted = y_true[desc_score_indices]
y_scores_sorted = y_scores[desc_score_indices]

# 2. Get unique thresholds
thresholds = np.unique(y_scores_sorted)[::-1]

# Lists to store ROC and PR points
tpr_list, fpr_list = [], []
precision_list, recall_list = [], []

P = np.sum(y_true)  # total positives
N = len(y_true) - P  # total negatives

# Lists to store results
rows = []

# 3. Loop through thresholds
for thresh in thresholds:
    # Predict positive if score >= threshold
    y_pred = (y_scores >= thresh).astype(int)
    
    TP = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    
    # ROC metrics
    TPR = TP / P if P > 0 else 0  # Recall
    FPR = FP / N if N > 0 else 0
    tpr_list.append(TPR)
    fpr_list.append(FPR)
    
    # PR metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 1
    recall = TP / P if P > 0 else 0
    precision_list.append(precision)
    recall_list.append(recall)

    rows.append({
        "Threshold": thresh,
        "TPR": round(TPR, 3),
        "FPR": round(FPR, 3),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3)
    })

# 4. Create DataFrame
df_metrics = pd.DataFrame(rows)

# 4. Add (0,0) for ROC and (1,0) for PR to complete curves
tpr_list = [0] + tpr_list + [1]
fpr_list = [0] + fpr_list + [1]
precision_list = [precision_list[0]] + precision_list
recall_list = [0] + recall_list

# 5. Calculate AUC manually (trapezoidal rule)
roc_auc = np.trapz(tpr_list, fpr_list)
pr_auc = np.trapz(precision_list, recall_list)

# 6. Plot ROC and PR curves
plt.figure(figsize=(12, 5))

# ROC
plt.subplot(1, 2, 1)
plt.plot(fpr_list, tpr_list, marker='o', label=f"ROC AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Manual ROC Curve")
plt.legend()

# PR
plt.subplot(1, 2, 2)
plt.plot(recall_list, precision_list, marker='o', label=f"PR AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Manual Precision-Recall Curve")
plt.legend()

plt.tight_layout()
plt.show()


In [0]:
print(df_metrics)

# PR AUC notes

### When is PR AUC Better than ROC AUC?

PR AUC is more informative when:

* The dataset is highly imbalanced (e.g., fraud detection, rare diseases).
* The positive class is rare and important.
* False positives and false negatives matter more than overall class separation.

ROC AUC can appear artificially high because a model can achieve a low FPR simply due to the large number of negative samples. PR AUC focuses on the quality of positive predictions.

### Example

For a dataset with **99% negatives** and **1% positives**:

* ROC AUC = **0.95** (appears excellent)
* PR AUC = **0.42** (reveals difficulty identifying positives accurately)

**Interpretation:**
The model separates classes reasonably well, but many positive predictions are incorrect, leading to low precision.

### Rule of Thumb

| Situation                      | ROC AUC        | PR AUC              |
| ------------------------------ | -------------- | ------------------- |
| Balanced classes               | ✅ Good         | ✅ Good              |
| Imbalanced classes             | ⚠️ Can Mislead | ✅ Preferred         |
| Rare, important positive class | ⚠️ Can Mislead | ✅ Best Choice       |
| Overall class separability     | ✅ Best         | ⚠️ Less Informative |

### Visual Intuition

* **ROC Curve:** Often looks strong on imbalanced data because FPR remains low.
* **PR Curve:** Drops quickly when false positives increase, providing a more realistic view of model performance.

### Key Takeaway

Use **ROC AUC** to measure overall class separation. Use **PR AUC** when the positive class is rare and business value depends on correctly identifying positives with minimal false alarms.

---

### PR AUC vs ROC AUC in 5 Points

1. **ROC AUC measures overall class separation**, while **PR AUC focuses on the performance of the positive class** by evaluating the trade-off between Precision and Recall.

2. **PR AUC is more suitable for imbalanced datasets** (e.g., fraud detection, churn prediction, disease diagnosis) because it is not influenced by the large number of negative cases.

3. **ROC AUC can appear misleadingly high in imbalanced datasets** since a low False Positive Rate is easy to achieve when negatives dominate, even if the model struggles to identify positives correctly.

4. **PR AUC provides a more realistic assessment of positive-class performance**, showing whether the model can find actual positives while minimizing false alarms.

5. **Rule of thumb:** Use **ROC AUC** for balanced datasets and overall separability assessment, but prefer **PR AUC** when the positive class is rare and business-critical, as it better reflects real-world model effectiveness.

---

**Will ROC AUC and PR AUC rank models the same way?**

**Not necessarily.** ROC AUC and PR AUC can produce different rankings, especially on imbalanced datasets.

### 5 Key Points

1. **ROC AUC measures overall class separability**, while **PR AUC focuses on positive-class performance** (Precision and Recall).

2. **For balanced datasets**, model rankings based on ROC AUC and PR AUC are often similar.

3. **For imbalanced datasets**, ROC AUC may rank a model highly even if it performs poorly on the minority class.

4. **PR AUC penalizes false positives more heavily**, so a model with slightly lower ROC AUC can have a higher PR AUC.

5. **Always compare both metrics**, but prioritize **PR AUC** when the positive class is rare and business-critical (fraud, churn, disease detection).

### Example

| Model | ROC AUC | PR AUC |
| ----- | ------- | ------ |
| A     | 0.96    | 0.42   |
| B     | 0.94    | 0.58   |
| C     | 0.92    | 0.65   |

**ROC Ranking:** A > B > C
**PR Ranking:** C > B > A

Although Model A separates classes best overall, Model C identifies positive cases more effectively with fewer false alarms, making it preferable in many imbalanced-class problems.

**Interview One-Liner:**
*"ROC AUC and PR AUC do not always follow the same trend; on imbalanced datasets, PR AUC often provides a more realistic ranking because it focuses on minority-class detection performance."*

---

### Best Threshold Selection on ROC AUC and PR AUC (5 Key Points)

1. **ROC Curve: Choose the point closest to the top-left corner (0,1).**

   * This point maximizes Recall (TPR) while minimizing False Positive Rate (FPR).
   * Distance formula:
     $[
     d=\sqrt{(1-TPR)^2+(FPR)^2}
     ]$
   * The threshold with the smallest distance is often considered optimal.

2. **ROC Curve: Maximum Youden's J Statistic.**

   * Common threshold selection criterion:
     [
     J = TPR - FPR
     ]
   * Select the threshold where (J) is highest.
   * Balances sensitivity and specificity.

3. **PR Curve: Maximum F1 Score.**

   * Since PR curves focus on Precision and Recall, the optimal threshold is often where:
     [
     F1=\frac{2\times Precision\times Recall}{Precision+Recall}
     ]
   * Highest F1 gives the best balance between Precision and Recall.

4. **PR Curve: Business-Driven Threshold Selection.**

   * For fraud, churn, and disease prediction, the "best" point depends on business goals.
   * If missing positives is costly → prioritize high Recall.
   * If false alarms are costly → prioritize high Precision.
   * The optimal threshold may not correspond to the highest F1 score.

5. **No Universal Best Threshold Exists.**

   * ROC and PR curves show performance across all thresholds.
   * The final threshold should be chosen based on:

     * Cost of False Positives vs False Negatives
     * Class imbalance
     * Business objectives
   * AUC measures overall model quality, while threshold selection determines operational performance.

### Interview One-Liner

**For ROC AUC, the optimal threshold is usually the point nearest the top-left corner or maximum Youden's J statistic. For PR AUC, the optimal threshold is commonly chosen using the maximum F1 score or based on business requirements for precision and recall trade-offs.**
